In [1]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [3]:
import wget
wget.download("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet")

100% [........................................................................] 71134255 / 71134255

'yellow_tripdata_2025-11.parquet'

In [4]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", True)\
    .parquet('yellow_tripdata_2025-11.parquet')

In [9]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df=df.repartition(4)

In [11]:
df.write.parquet('hwdata/pq/2025/11/')

In [13]:
df = spark.read.parquet('hwdata/pq/2025/11/')

In [14]:
from pyspark.sql import functions as F

In [26]:
df1= df\
.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime))\
.withColumn('dropoff_date',F.to_date(df.tpep_dropoff_datetime))\
.select('pickup_date','dropoff_date','VendorID','PULocationID','DOLocationID')

In [27]:
df1.show()

+-----------+------------+--------+------------+------------+
|pickup_date|dropoff_date|VendorID|PULocationID|DOLocationID|
+-----------+------------+--------+------------+------------+
| 2025-11-02|  2025-11-02|       2|         186|         230|
| 2025-11-06|  2025-11-06|       2|         164|         237|
| 2025-11-07|  2025-11-07|       2|         186|         161|
| 2025-11-09|  2025-11-09|       2|          68|         246|
| 2025-11-03|  2025-11-03|       2|          90|         164|
| 2025-11-06|  2025-11-06|       2|         132|         107|
| 2025-11-07|  2025-11-07|       1|         237|         237|
| 2025-11-07|  2025-11-07|       1|         170|         141|
| 2025-11-05|  2025-11-05|       2|         237|          74|
| 2025-11-04|  2025-11-04|       2|         239|         263|
| 2025-11-05|  2025-11-05|       2|         100|         234|
| 2025-11-06|  2025-11-06|       2|          48|         249|
| 2025-11-01|  2025-11-01|       2|          48|         238|
| 2025-1

In [29]:
df1.createOrReplaceTempView('trips_dated')

In [37]:
trips_15nov= spark.sql("""
SELECT count(*), pickup_date
FROM trips_dated
WHERE pickup_date ='2025-11-15'
GROUP BY pickup_date
""").show()

+--------+-----------+
|count(1)|pickup_date|
+--------+-----------+
|  162604| 2025-11-15|
+--------+-----------+



In [51]:
df2= df\
.withColumn('trip_duration_in_hours', F.timestamp_diff('hour','tpep_pickup_datetime','tpep_dropoff_datetime'))\
.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime))\
.withColumn('dropoff_date',F.to_date(df.tpep_dropoff_datetime))\
.select('pickup_date','dropoff_date','VendorID','PULocationID','DOLocationID','trip_duration_in_hours')

In [43]:
df2.show()

+-----------+------------+--------+------------+------------+----------------------+
|pickup_date|dropoff_date|VendorID|PULocationID|DOLocationID|trip_duration_in_hours|
+-----------+------------+--------+------------+------------+----------------------+
| 2025-11-02|  2025-11-02|       2|         186|         230|                     0|
| 2025-11-06|  2025-11-06|       2|         164|         237|                     0|
| 2025-11-07|  2025-11-07|       2|         186|         161|                     0|
| 2025-11-09|  2025-11-09|       2|          68|         246|                     0|
| 2025-11-03|  2025-11-03|       2|          90|         164|                     0|
| 2025-11-06|  2025-11-06|       2|         132|         107|                     1|
| 2025-11-07|  2025-11-07|       1|         237|         237|                     0|
| 2025-11-07|  2025-11-07|       1|         170|         141|                     0|
| 2025-11-05|  2025-11-05|       2|         237|          74|    

In [44]:
df2.createOrReplaceTempView('trips_duration')

In [52]:
longest_trip= spark.sql("""
SELECT trip_duration_in_hours,pickup_date,dropoff_date
FROM trips_duration
ORDER BY trip_duration_in_hours DESC
LIMIT 1
""")

In [54]:
longest_trip.show()

+----------------------+-----------+------------+
|trip_duration_in_hours|pickup_date|dropoff_date|
+----------------------+-----------+------------+
|                    90| 2025-11-26|  2025-11-30|
+----------------------+-----------+------------+



In [56]:
wget.download('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')

100% [..............................................................................] 12331 / 12331

'taxi_zone_lookup.csv'

In [58]:
df_zones = spark.read.option("header", True).option("inferSchema", True).csv("taxi_zone_lookup.csv")

In [59]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [60]:
df_zones.write.mode("overwrite").parquet('hwdata/pq/zones/')

In [61]:
df_zones= spark.read.parquet('hwdata/pq/zones/')

In [62]:
df_join= df1.join(df_zones, df1.PULocationID == df_zones.LocationID)

In [64]:
df_join= df_join.drop('LocationID','DOLocationID','VendorID','Borough','service_zone')

In [65]:
df_join.show()


+-----------+------------+------------+--------------------+
|pickup_date|dropoff_date|PULocationID|                Zone|
+-----------+------------+------------+--------------------+
| 2025-11-02|  2025-11-02|         186|Penn Station/Madi...|
| 2025-11-06|  2025-11-06|         164|       Midtown South|
| 2025-11-07|  2025-11-07|         186|Penn Station/Madi...|
| 2025-11-09|  2025-11-09|          68|        East Chelsea|
| 2025-11-03|  2025-11-03|          90|            Flatiron|
| 2025-11-06|  2025-11-06|         132|         JFK Airport|
| 2025-11-07|  2025-11-07|         237|Upper East Side S...|
| 2025-11-07|  2025-11-07|         170|         Murray Hill|
| 2025-11-05|  2025-11-05|         237|Upper East Side S...|
| 2025-11-04|  2025-11-04|         239|Upper West Side S...|
| 2025-11-05|  2025-11-05|         100|    Garment District|
| 2025-11-06|  2025-11-06|          48|        Clinton East|
| 2025-11-01|  2025-11-01|          48|        Clinton East|
| 2025-11-03|  2025-11-0

In [66]:
df_join.createOrReplaceTempView('trips_zones')

In [73]:
Least_freq_zone = spark.sql("""
SELECT count(*) AS frequency, Zone
FROM trips_zones
GROUP BY Zone
ORDER BY frequency ASC
LIMIT 3
""")

In [74]:
Least_freq_zone.show()

+---------+--------------------+
|frequency|                Zone|
+---------+--------------------+
|        1|Eltingville/Annad...|
|        1|       Arden Heights|
|        1|Governor's Island...|
+---------+--------------------+

